# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haroonrana330/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice My baseline (Week 4) was a hand-built rule: a weighted formula (40% visibility, 30% freshness, 25% position opportunity, 5% content depth) that scores pages 0–1, with no learning involved — the weights were chosen by me, not learned from data. For Week 5, I'm building a Logistic Regression classifier as my primary model, with Random Forest as a stronger comparison model. Target: is_declining_label — whether a page's traffic trend is declining (1) or not (0). This label already exists in my data, derived from trend_direction .Why Logistic Regression first: It's simple, interpretable, and gives me clear coefficients I can explain — a fair, honest next step up from a handpicked rule, since it learns weights from data instead of me guessing them. Why Random Forest second: It can capture non-linear relationships and interactions between signals that Logistic Regression can't, so it tells me whether the extra complexity is actually worth it — or whether my simple rule/logistic model was already good enough. Leakage precaution: I am excluding trend_direction and trend_pct from the model's input features, since the label itself was derived directly from trend_direction — including it would let the model "cheat" by looking at the answer.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design I'm using a grouped train/test split by client_id , not a random row-level split. Since multiple pages can belong to the same client, a random split could put pages from the same client in both train and test, letting the model "see" client-specific patterns during training that leak into testing. Splitting by client ensures the model is tested on entirely unseen clients — a more honest measure of real-world performance.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Clone the repo so local file paths work
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

if not os.path.exists("/content/flyrank-ml"):
    !git clone https://github.com/haroonrana330/flyrank-ml.git /content/flyrank-ml

os.chdir("/content/flyrank-ml")
print("Current directory:", os.getcwd())

# Load data
url = "https://raw.githubusercontent.com/haroonrana330/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

# Recreate the label
df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

# --- Recreate the Week-4 baseline score inline (file wasn't committed to repo) ---
numeric_columns = ["impressions_90d", "clicks_90d", "sessions_90d", "content_age_days",
                    "days_since_last_update", "word_count", "avg_position", "ctr",
                    "engagement_rate", "scroll_rate"]
for col in numeric_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

def percentile_rank(series):
    return series.rank(pct=True).fillna(0)

def normalize(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.0, index=series.index)
    return (series - mn) / (mx - mn)

df["visibility_score"] = percentile_rank(np.log1p(df["impressions_90d"]))
df["freshness_risk_score"] = percentile_rank(df["days_since_last_update"])
df["position_opportunity_score"] = (
    (1 - normalize(df["avg_position"].clip(lower=1, upper=50)))
    * df["visibility_score"] * (df["avg_position"] > 0).astype(int)
)
df["depth_gap_score"] = (1 - percentile_rank(df["word_count"])) * df["visibility_score"]
df["baseline_refresh_score"] = (
    0.40 * df["visibility_score"] + 0.30 * df["freshness_risk_score"]
    + 0.25 * df["position_opportunity_score"] + 0.05 * df["depth_gap_score"]
).clip(0, 1)
# --- End baseline recreation ---

# Features for the new model
feature_cols = ["search_volume", "competition", "cpc", "word_count", "char_count",
                 "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"]
feature_cols = [c for c in feature_cols if c in df.columns]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(df[col].median())

X = df[feature_cols]
y = df["is_declining_label"]
groups = df["client_id"]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test = df.iloc[test_idx]["baseline_refresh_score"]

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Unique clients in train:", df.iloc[train_idx]["client_id"].nunique())
print("Unique clients in test:", df.iloc[test_idx]["client_id"].nunique())

Current directory: /content/flyrank-ml
Train rows: 23837 | Test rows: 6163
Unique clients in train: 25
Unique clients in test: 7


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Training and comparing against the baseline
I'm training Logistic Regression and Random Forest on the same train/test split, then comparing both models' ROC-AUC score against my Week-4 baseline's baseline_refresh_score on the exact same test set. ROC-AUC lets me compare a rule-based score and a trained model's probability output on the same scale, since both just need to rank pages correctly regardless of exact score values.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Scale features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Logistic Regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)
log_reg_probs = log_reg.predict_proba(X_test_scaled)[:, 1]
log_reg_auc = roc_auc_score(y_test, log_reg_probs)

# Random Forest (no scaling needed)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_probs)

# Baseline comparison (same test set, same metric)
baseline_auc = roc_auc_score(y_test, baseline_test.fillna(baseline_test.median()))

print("MODEL vs BASELINE COMPARISON (ROC-AUC, same test set)")
print("=" * 60)
print(f"Baseline (Week-4 rule):     {baseline_auc:.4f}")
print(f"Logistic Regression:       {log_reg_auc:.4f}")
print(f"Random Forest:              {rf_auc:.4f}")

MODEL vs BASELINE COMPARISON (ROC-AUC, same test set)
Baseline (Week-4 rule):     0.4979
Logistic Regression:       0.5407
Random Forest:              0.5655


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

My Random Forest scored 0.5655 ROC-AUC versus my Week-4 baseline's 0.4979 — a real, meaningful improvement, since the baseline was barely better than random guessing (0.5) at this specific task.
Looking at the confusion matrix, the model is heavily skewed toward predicting "declining." It correctly catches 91% of pages that are actually declining (recall for class 1), but only 15% of pages that are actually stable (recall for class 0) — meaning it mislabels a large number of stable pages as declining (2,554 false positives) far more often than it misses real declines (285 false negatives). In practice, this means the model errs on the side of caution: it would rather over-flag pages for review than risk missing a real decline, but that comes at the cost of a lot of unnecessary flags on pages that didn't actually need attention.
The feature importance results show avg_position is by far the strongest driver of the model's decisions, accounting for 46% of its total reliance — more than three times any other single feature. word_count (16%) and char_count (11%) are the next most influential, together with avg_position making up about 73% of what the model actually uses. This is notably different from my Week-4 baseline rule, which weighted visibility (40%) and freshness (30%) most heavily and gave content depth only a 5% weight — the trained model, left to find its own patterns, arrived at a very different picture of what actually predicts decline, leaning much harder on ranking position than on visibility or freshness.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.metrics import confusion_matrix, classification_report

# Use Random Forest predictions (the stronger model) for error inspection
rf_preds = (rf_probs >= 0.5).astype(int)

print("CONFUSION MATRIX (Random Forest, threshold 0.5)")
print("=" * 60)
cm = confusion_matrix(y_test, rf_preds)
print(cm)
print()
print("Rows = actual, Columns = predicted | [0,0]=TN [0,1]=FP [1,0]=FN [1,1]=TP")
print()

print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, rf_preds))

# Feature importance from Random Forest — what actually drove the predictions
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("FEATURE IMPORTANCE (Random Forest)")
print("=" * 60)
print(importances)

CONFUSION MATRIX (Random Forest, threshold 0.5)
[[ 460 2554]
 [ 285 2864]]

Rows = actual, Columns = predicted | [0,0]=TN [0,1]=FP [1,0]=FN [1,1]=TP

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.62      0.15      0.24      3014
           1       0.53      0.91      0.67      3149

    accuracy                           0.54      6163
   macro avg       0.57      0.53      0.46      6163
weighted avg       0.57      0.54      0.46      6163

FEATURE IMPORTANCE (Random Forest)
avg_position       0.462813
word_count         0.159855
char_count         0.105700
search_volume      0.094448
ctr                0.072974
scroll_rate        0.058609
engagement_rate    0.020428
competition        0.014898
cpc                0.005746
ai_traffic_pct     0.004530
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.